# Appendix A - Data representation and sampling

*This notebook contains all the sample code in Appendix A.*

## Outline

- [Sampling observations for machine learning](#Sampling)
- [Resampling](#Resampling)

Machine-learning algorithms operate on numerical representations of observations. The quality of these representations, and the procedure used to select the observations, strongly influence the reliability of the resulting model.

## Sampling observations for machine learning <a id="Sampling"></a>

In statistics, a **population** is the complete set of observations of interest, whereas a **sample** is the subset used for analysis. A learning algorithm can generalize only when the available sample is sufficiently representative of the population and of the conditions expected after deployment.

Random sampling is useful when observations can reasonably be regarded as independent and identically distributed. This assumption may be inappropriate when data contain groups, temporal dependence, spatial dependence, or repeated measurements. Convenience samples and poorly defined inclusion criteria can introduce sampling bias that no learning algorithm can remove.

The splitting strategy must reflect the structure of the data:
- **stratified splitting** approximately preserves class proportions in classification problems;
- **grouped splitting** keeps all observations from the same subject, device, location, or experiment in one subset;
- **time-ordered splitting** trains on earlier observations and evaluates on later ones, avoiding random shuffling of future information into the training set.

When observations are duplicated or strongly related, the split must be performed before augmentation or preprocessing that could transfer information between subsets.

The following NumPy function performs a simple stratified train--test split without using external machine-learning libraries:

In [1]:
import numpy as np

def stratified_split(X, y, test_fraction=0.2, seed=None):
    """Return stratified train and test subsets."""
    X = np.asarray(X)
    y = np.asarray(y)

    rng = np.random.default_rng(seed)
    train_indices = []
    test_indices = []

    for label in np.unique(y):
        indices = np.flatnonzero(y == label)
        rng.shuffle(indices)

        n_test = max(1, int(round(test_fraction * len(indices))))
        if n_test >= len(indices):
            n_test = len(indices) - 1

        test_indices.extend(indices[:n_test])
        train_indices.extend(indices[n_test:])

    train_indices = np.asarray(train_indices)
    test_indices = np.asarray(test_indices)
    rng.shuffle(train_indices)
    rng.shuffle(test_indices)

    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]

## Resampling <a id="Resampling"></a>

**Resampling** constructs new datasets from the available observations. It is used to estimate uncertainty, compare models, and reduce dependence on a particular split.

In ***$k$-fold cross-validation***, the data are partitioned into $k$ *folds*. Each fold is used once for validation while the remaining folds are used for training. All preprocessing and model selection operations must be repeated independently inside each training fold.

The **bootstrap** draws $N$ observations with replacement from a dataset of size $N$. Some observations may appear several times, while others are omitted. Repeating this procedure provides an empirical distribution of an estimator or performance measure. A basic NumPy implementation is:

In [2]:
import numpy as np

def bootstrap_sample(X, y=None, seed=None):
    """Draw one bootstrap sample with replacement."""
    X = np.asarray(X)
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(X), size=len(X))

    if y is None:
        return X[indices]

    y = np.asarray(y)

    return X[indices], y[indices]

Cross-validation and bootstrap estimates answer different questions and are not interchangeable. Cross-validation is commonly used to estimate predictive performance and tune models, whereas the bootstrap is especially useful for assessing sampling variability. Neither method corrects an unrepresentative or biased original dataset.